In [1]:
from irrigator.forecasts.arome_processing import (
    sync_static,
    sync_forecast,
    sync_dynamic,
    get_available_coverages,
    parse_coverage_id,
    find_available_run_dates,
    load_arome_daily_cache,
    ensure_dirs,
    DAILY_DIR,
    RAW_DIR,
)
from collections import defaultdict
import os

REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)


results = sync_static(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

results_dyn = sync_dynamic(overwrite=False)


results = sync_forecast(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

    

2026-08-14 20:32:52,071 Syncing static archive...
2026-08-14 20:33:06,150 Sync complete: 5 available, 5 cached, 0 fetched, 0 errors
2026-08-14 20:33:06,152 Syncing dynamic archive...


  ✓ 2026-08-10: cached
  ✓ 2026-08-11: cached
  ✓ 2026-08-12: cached
  ✓ 2026-08-13: cached
  ✓ 2026-08-14: cached


2026-08-14 20:33:28,101 [dynamic] 2026-08-14T15: done
2026-08-14 20:33:34,561 [dynamic] arome_daily_2026-08-10.nc → 8 windows, 23.7 MB
2026-08-14 20:33:38,570 [dynamic] arome_daily_2026-08-11.nc → 8 windows, 23.4 MB
2026-08-14 20:33:42,604 [dynamic] arome_daily_2026-08-12.nc → 8 windows, 23.3 MB
2026-08-14 20:33:46,673 [dynamic] arome_daily_2026-08-13.nc → 8 windows, 23.6 MB
2026-08-14 20:33:49,926 [dynamic] arome_daily_2026-08-14.nc → 6 windows, 24.3 MB
2026-08-14 20:33:49,930 Syncing forecast archive...
2026-08-14 20:34:07,596 Sync complete: 5 available, 5 cached, 0 fetched, 0 errors


  ✓ 2026-08-10: cached
  ✓ 2026-08-11: cached
  ✓ 2026-08-12: cached
  ✓ 2026-08-13: cached
  ✓ 2026-08-14: cached


# IFS

In [2]:
import os
import logging
from datetime import date, timedelta
from pathlib import Path
from irrigator.ingestion.ifs_ens_client import run_ifs_pipeline, load_ifs_daily

# Today only (default)
# daily_paths = run_ifs_pipeline()

# Or a date range:
daily_paths = run_ifs_pipeline(
    start=date(2026, 8, 5),
    end=date.today(),
    keep_raw=False,  # delete GRIBs after processing (default)
    #split_params=False
)

print(f"\nProcessed {len(daily_paths)} runs:")
for p in daily_paths:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.0f} MB)")


2026-08-14 20:34:07,631 IFS ENS 2026-08-05 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-05_00z.nc
2026-08-14 20:34:38,589 IFS ENS 2026-08-06 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-06_00z.nc
2026-08-14 20:35:09,476 IFS ENS 2026-08-07 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-07_00z.nc
2026-08-14 20:35:40,464 IFS ENS 2026-08-08 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-08_00z.nc
2026-08-14 20:36:11,421 IFS ENS 2026-08-09 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-09_00z.nc
2026-08-14 20:36:42,401 IFS ENS 2026-08-10 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-10_00z.nc
2026-08-14 20:37:13,398 IFS ENS 2026-08-11 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-08-11_00z.nc
2026-08-14 20:37:43,399 [2026-08-12 00Z] Downloading...
2026-08-14 20:37:43,418 Downloading IFS ENS (bulk, source=ecmwf): 2026-08-12 00Z, 61 steps, 7 params
2026-08-14 20:37:43,419 

<multiple>:   0%|          | 0.00/16.3G [00:00<?, ?B/s]

2026-08-14 21:04:42,875 Downloaded IFS ENS: data/raw/ifs_ens/ifs_ens_2026-08-12_00z.grib2 (17449.9 MB)
2026-08-14 21:04:42,876 [2026-08-12 00Z] Opening and slicing to bbox...


By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.


2026-08-14 21:05:53,900 [2026-08-12 00Z] Processing to daily...
2026-08-14 21:13:10,653 IFS ENS daily: 16 days, 50 members, 8 variables
2026-08-14 21:13:12,660 Saved IFS ENS daily: data/processed/ifs_ens/ifs_daily_2026-08-12_00z.nc (47.1 MB, 50 members, 16 days)
2026-08-14 21:13:14,034 [2026-08-12 00Z] Deleted raw GRIB (17450 MB freed)
2026-08-14 21:13:45,427 [2026-08-13 00Z] Downloading...
2026-08-14 21:13:45,429 Downloading IFS ENS (bulk, source=ecmwf): 2026-08-13 00Z, 61 steps, 7 params
2026-08-14 21:14:17,700 Downloading <multiple>


<multiple>:   0%|          | 0.00/16.2G [00:00<?, ?B/s]

2026-08-14 21:37:09,972 Downloaded IFS ENS: data/raw/ifs_ens/ifs_ens_2026-08-13_00z.grib2 (17382.3 MB)
2026-08-14 21:37:09,973 [2026-08-13 00Z] Opening and slicing to bbox...
2026-08-14 21:38:23,167 [2026-08-13 00Z] Processing to daily...
2026-08-14 21:45:42,094 IFS ENS daily: 16 days, 50 members, 8 variables
2026-08-14 21:45:43,927 Saved IFS ENS daily: data/processed/ifs_ens/ifs_daily_2026-08-13_00z.nc (47.3 MB, 50 members, 16 days)
2026-08-14 21:45:45,050 [2026-08-13 00Z] Deleted raw GRIB (17382 MB freed)
2026-08-14 21:46:16,433 [2026-08-14 00Z] Downloading...
2026-08-14 21:46:16,434 Downloading IFS ENS (bulk, source=ecmwf): 2026-08-14 00Z, 61 steps, 7 params
2026-08-14 21:46:17,005 Recovering from HTTP error [429 Too Many Requests], attempt 1 of 500
2026-08-14 21:46:17,006 Retrying in 120 seconds
2026-08-14 21:48:22,512 Retrying now...
2026-08-14 21:48:22,664 Recovering from HTTP error [429 Too Many Requests], attempt 2 of 500
2026-08-14 21:48:22,665 Retrying in 120 seconds
2026-08-

<multiple>:   0%|          | 0.00/16.2G [00:00<?, ?B/s]

2026-08-14 22:01:25,338 Recovering from HTTP error [429 Too Many Requests], attempt 1 of 500
2026-08-14 22:01:25,338 Retrying in 120 seconds
2026-08-14 22:03:30,825 Retrying now...
2026-08-14 22:19:33,173 Downloaded IFS ENS: data/raw/ifs_ens/ifs_ens_2026-08-14_00z.grib2 (17363.1 MB)
2026-08-14 22:19:33,174 [2026-08-14 00Z] Opening and slicing to bbox...
2026-08-14 22:20:43,726 [2026-08-14 00Z] Processing to daily...
2026-08-14 22:27:59,364 IFS ENS daily: 16 days, 50 members, 8 variables
2026-08-14 22:28:01,172 Saved IFS ENS daily: data/processed/ifs_ens/ifs_daily_2026-08-14_00z.nc (47.4 MB, 50 members, 16 days)
2026-08-14 22:28:02,499 [2026-08-14 00Z] Deleted raw GRIB (17363 MB freed)
2026-08-14 22:28:02,511 IFS pipeline complete: 10/10 runs processed



Processed 10 runs:
  ifs_daily_2026-08-05_00z.nc  (46 MB)
  ifs_daily_2026-08-06_00z.nc  (46 MB)
  ifs_daily_2026-08-07_00z.nc  (46 MB)
  ifs_daily_2026-08-08_00z.nc  (46 MB)
  ifs_daily_2026-08-09_00z.nc  (46 MB)
  ifs_daily_2026-08-10_00z.nc  (46 MB)
  ifs_daily_2026-08-11_00z.nc  (47 MB)
  ifs_daily_2026-08-12_00z.nc  (47 MB)
  ifs_daily_2026-08-13_00z.nc  (47 MB)
  ifs_daily_2026-08-14_00z.nc  (47 MB)


# SEAS5 hindcast archive (one-time per initialization month)

The seasonal bias correction needs the ECMWF SEAS5 1993–2016 retrospective
forecasts for the same initialization month as the operational forecast. The
download is intentionally opt-in because it is a sizeable one-time archive.

You also need processed France-wide ERA5-Land daily files covering the historical
period used for analog matching; the new PCA code reads those annual files lazily.

In [ ]:
from datetime import date
from irrigator.ingestion.cds_client import fetch_seas5_hindcasts

RUN_SEAS5_HINDCAST_ARCHIVE = True
SEAS5_INIT_MONTH = date.today().month

if RUN_SEAS5_HINDCAST_ARCHIVE:
    paths = fetch_seas5_hindcasts(
        init_month=SEAS5_INIT_MONTH,
        start_year=1993,
        end_year=2016,
        raw_dir=Path("data/raw"),
        overwrite=False,
    )
    print(f"SEAS5 hindcasts ready: {len(paths)} initialization files")
else:
    print("SEAS5 hindcast archive skipped. Set RUN_SEAS5_HINDCAST_ARCHIVE=True once when needed.")


In [2]:
from irrigator.ingestion.cds_client import _seas5_output_path, FRANCE_BBOX, DEFAULT_RAW_DIR, BBoxWGS84
from pathlib import Path
import xarray as xr

from datetime import date
import numpy as np


def date_to_datetime64(d: date) -> np.datetime64:
    return np.datetime64(d, "ns")

def fetch_seas5(
    year: int,
    month: int,
    ref_file: str,
    *,
    bounding_box: BBoxWGS84 = FRANCE_BBOX,
    raw_dir: str | Path = DEFAULT_RAW_DIR,
) -> Path:
    """Download SEAS5 seasonal forecast initialised at year/month.

    Downloads all lead times (1-6 months) and all ensemble members.

    Parameters
    ----------
    year, month : initialisation date of the forecast
    bounding_box : spatial extent (default: France metropolitan)
    raw_dir : output directory
    overwrite : re-download even if file exists

    Returns
    -------
    Path to downloaded NetCDF.
    """
    raw_dir = Path(raw_dir)
    out_path = _seas5_output_path(raw_dir, year, month)
    ds_ref = xr.open_dataset(ref_file)
    ds_ref.sel(forecast_reference_time=date_to_datetime64(date(year, month,1)), longitude=slice(bounding_box.west, bounding_box.east), latitude=slice(bounding_box.north, bounding_box.south)).to_netcdf(out_path)
    return out_path

ref_file="/home/mbaldacchino/data/seas5_archive_3.nc"

for year in range(2010, 2017):
    for month in range(1, 13):
        out_path = fetch_seas5(year, month, ref_file=ref_file)
        print(f"Downloaded SEAS5 {year}-{month:02d} to {out_path}")

Downloaded SEAS5 2010-01 to data/raw/seas5/seas5_2010_01.nc
Downloaded SEAS5 2010-02 to data/raw/seas5/seas5_2010_02.nc
Downloaded SEAS5 2010-03 to data/raw/seas5/seas5_2010_03.nc
Downloaded SEAS5 2010-04 to data/raw/seas5/seas5_2010_04.nc
Downloaded SEAS5 2010-05 to data/raw/seas5/seas5_2010_05.nc
Downloaded SEAS5 2010-06 to data/raw/seas5/seas5_2010_06.nc
Downloaded SEAS5 2010-07 to data/raw/seas5/seas5_2010_07.nc
Downloaded SEAS5 2010-08 to data/raw/seas5/seas5_2010_08.nc
Downloaded SEAS5 2010-09 to data/raw/seas5/seas5_2010_09.nc
Downloaded SEAS5 2010-10 to data/raw/seas5/seas5_2010_10.nc
Downloaded SEAS5 2010-11 to data/raw/seas5/seas5_2010_11.nc
Downloaded SEAS5 2010-12 to data/raw/seas5/seas5_2010_12.nc
Downloaded SEAS5 2011-01 to data/raw/seas5/seas5_2011_01.nc
Downloaded SEAS5 2011-02 to data/raw/seas5/seas5_2011_02.nc
Downloaded SEAS5 2011-03 to data/raw/seas5/seas5_2011_03.nc
Downloaded SEAS5 2011-04 to data/raw/seas5/seas5_2011_04.nc
Downloaded SEAS5 2011-05 to data/raw/sea